In [1]:
import json
from pathlib import Path
from math import sqrt
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt 
from sklearn.preprocessing import StandardScaler
import hdbscan
import seaborn as sns
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
from sklearn.cluster import KMeans
from scipy.signal import savgol_filter, find_peaks
from scipy.stats import mode
from collections import defaultdict
from sklearn.preprocessing import normalize
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from scipy.signal import savgol_filter, find_peaks
from sklearn.cluster import MeanShift, estimate_bandwidth
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import normalize
import cv2
import os
from collections import defaultdict

In [ ]:
"""
load_annotations.py
────────────────────
Loads rat_movement.json and builds a numpy array of shape:
  (n_frames, n_bodyparts, n_coordinates)  →  e.g. (550, 7, 2)

Also creates a labelled xarray DataArray for convenient named-axis access.

Requirements:
  pip install numpy xarray
"""

import json
import numpy as np
import xarray as xr

# ── load JSON ─────────────────────────────────────────────────────────────────
with open("rat_movement_large.json") as f:
    data = json.load(f)

annotations = data["annotations"]

# ── extract ordered axes ──────────────────────────────────────────────────────
n_frames    = len(annotations)
frame_ids   = [str(i) for i in range(n_frames)]           # "0", "1", …
body_parts  = list(annotations["0"].keys())               # preserves insertion order
n_bodyparts = len(body_parts)
coords_axes = ["x", "y"]
n_coords    = len(coords_axes)

# ── build numpy array  (n_frames, n_bodyparts, n_coordinates) ─────────────────
arr = np.zeros((n_frames, n_bodyparts, n_coords), dtype=np.float32)

for fi, fid in enumerate(frame_ids):
    for bi, bp in enumerate(body_parts):
        arr[fi, bi, :] = annotations[fid][bp]   # [x, y]

print("numpy array shape:", arr.shape)           # (550, 7, 2)
print("dtype           :", arr.dtype)

# ── wrap in xarray for labelled access (optional but handy) ───────────────────
da = xr.DataArray(
    arr,
    dims=["frame", "bodypart", "coord"],
    coords={
        "frame":    np.arange(n_frames),
        "bodypart": body_parts,
        "coord":    coords_axes,
    },
    name="keypoints",
)

print("\nxarray DataArray:\n", da)

# ── example accesses ──────────────────────────────────────────────────────────
# All (x, y) positions of the head across every frame  → shape (n_frames, 2)
head_xy = da.sel(bodypart="head").values
print("\nhead x/y — first 5 frames:\n", head_xy[:5])

# x-coordinate of every body part at frame 0  → shape (n_bodyparts,)
frame0_x = da.sel(frame=0, coord="x").values
print("\nframe 0 — x coords per body part:")
for bp, x in zip(body_parts, frame0_x):
    print(f"  {bp:20s} {x:.1f}")

In [ ]:
#load experimental data #RUN THIS CELL NOT THE ONE ABOVE
import json
import numpy as np
import xarray as xr
from pathlib import Path

# ── just point to your folder ─────────────────────────────────────────────────
json_folder = Path(r"C:\Users\ariAccount\Desktop\jackie_data_willdeletelater\actual_final_annotations")
json_files  = list(json_folder.glob("*.json"))
print(f"Found {len(json_files)} json files")

all_arrays = []
body_parts = None

for json_file in json_files:
    with open(json_file) as f:
        data = json.load(f)

    annotations = data["annotations"]
    frame_ids   = list(annotations.keys())
    n_frames    = len(frame_ids)
    first_key   = frame_ids[0]

    if body_parts is None:
        body_parts = list(annotations[first_key].keys())
    else:
        assert list(annotations[first_key].keys()) == body_parts, \
            f"Body parts mismatch in {json_file}"

    n_bodyparts = len(body_parts)
    arr = np.zeros((n_frames, n_bodyparts, 2), dtype=np.float32)

    for fi, fid in enumerate(frame_ids):
        for bi, bp in enumerate(body_parts):
            arr[fi, bi, :] = annotations[fid][bp]

    all_arrays.append(arr)
    print(f"Loaded {json_file.name} — {n_frames} frames")

# ── concatenate everything into one array ─────────────────────────────────────
arr_all = np.concatenate(all_arrays, axis=0)

da = xr.DataArray(
    arr_all,
    dims=["frame", "bodypart", "coord"],
    coords={
        "frame":    np.arange(len(arr_all)),
        "bodypart": body_parts,
        "coord":    ["x", "y"],
    },
    name="keypoints",
)

print("\nDataArray shape:", da.shape)

Found 12 json files
Loaded 2026-05-11 10-45-46_bottom_left.json — 312589 frames
Loaded 2026-05-11 10-45-46_bottom_right.json — 314117 frames
Loaded 2026-05-11 10-45-46_top_left.json — 313777 frames
Loaded 2026-05-11 10-45-46_top_right.json — 314264 frames
Loaded 2026-05-11 10-46-09_bottom_right.json — 317041 frames
Loaded 2026-05-11 10-46-09_top_left.json — 316845 frames
Loaded 2026-05-12 11-26-37_bottom_left.json — 307915 frames
Loaded 2026-05-12 11-26-37_bottom_right.json — 308734 frames
Loaded 2026-05-12 11-26-37_top_left.json — 298745 frames
Loaded 2026-05-12 11-26-37_top_right.json — 308150 frames
Loaded 2026-05-12 11-27-00_bottom_right.json — 322637 frames
Loaded 2026-05-12 11-27-00_top_left.json — 322488 frames

DataArray shape: (3757302, 7, 2)


In [ ]:
def preprocess(raw_sequence):
    # compute speed from raw arena movement BEFORE centering
    center_raw  = raw_sequence[:, 1, :]                        # (T, 2)
    center_diff = np.diff(center_raw, axis=0)                  # (T-1, 2)
    center_diff = np.concatenate([center_diff[:1], center_diff], axis=0)  # (T, 2)
    speed       = np.linalg.norm(center_diff, axis=-1)         # (T,)
    print(f"Speed stats: mean={speed.mean():.3f}, std={speed.std():.3f}, max={speed.max():.3f}")
    speed       = np.tile(speed[:, None, None], (1, 7, 1))     # (T, 7, 1)

    # center and heading-align
    centroid = raw_sequence[:, 1:2, :]
    centered = raw_sequence - centroid

    head  = centered[:, 0, :]
    angle = np.arctan2(head[:, 1], head[:, 0])
    cos_a = np.cos(-angle)
    sin_a = np.sin(-angle)

    x = centered[:, :, 0]
    y = centered[:, :, 1]
    x_rot = x * cos_a[:, None] - y * sin_a[:, None]
    y_rot = x * sin_a[:, None] + y * cos_a[:, None]
    aligned = np.stack([x_rot, y_rot], axis=-1)                # (T, 7, 2)

    # velocity of aligned joints (captures gait/paw swing)
    vel = np.diff(aligned, axis=0)
    vel = np.concatenate([vel[:1], vel], axis=0)               # (T, 7, 2)

    # angular velocity (captures turning)
    ang_vel = np.diff(angle, axis=0)
    ang_vel = np.concatenate([ang_vel[:1], ang_vel])
    ang_vel = np.tile(ang_vel[:, None, None], (1, 7, 1))       # (T, 7, 1)

    return np.concatenate([aligned, vel, ang_vel, speed], axis=-1)  # (T, 7, 6)

raw_processed = preprocess(da.values)


#raw_processed = preprocess(da.values)

In [ ]:
#RUN THIS CELL NOT THE ONE ABOVE
def preprocess_exp(raw_sequence):
    # absolute positions — location is meaningful
    positions = raw_sequence.copy()                # (T, 7, 2)

    # velocity per joint
    vel = np.diff(positions, axis=0)
    vel = np.concatenate([vel[:1], vel], axis=0)   # (T, 7, 2)

    # speed from center joint BEFORE any transformation
    center_diff = np.diff(raw_sequence[:, 1, :], axis=0)
    center_diff = np.concatenate([center_diff[:1], center_diff], axis=0)
    speed       = np.linalg.norm(center_diff, axis=-1)
    speed       = np.tile(speed[:, None, None], (1, 7, 1))  # (T, 7, 1)

    # angular velocity from head-to-center angle
    centered = raw_sequence - raw_sequence[:, 1:2, :]
    head     = centered[:, 0, :]
    angle    = np.arctan2(head[:, 1], head[:, 0])
    ang_vel  = np.diff(angle, axis=0)
    ang_vel  = np.concatenate([ang_vel[:1], ang_vel])
    ang_vel  = np.tile(ang_vel[:, None, None], (1, 7, 1))  # (T, 7, 1)

    return np.concatenate([positions, vel, ang_vel, speed], axis=-1)  # (T, 7, 6)

raw_processed= preprocess_exp(da.values)

In [8]:
n_frames, n_joints, n_coords = raw_processed.shape
da_scaled = np.zeros_like(raw_processed)
scalers = []

for j in range(n_joints):
    joint_scalers = []
    for c in range(n_coords):
        scaler = StandardScaler()
        da_scaled[:, j, c] = scaler.fit_transform(
            raw_processed[:, j, c].reshape(-1, 1)
        ).squeeze()
        joint_scalers.append(scaler)
    scalers.append(joint_scalers)

raw_processed = da_scaled

In [9]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [10]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.preprocessing import normalize
from sklearn.cluster import MeanShift, estimate_bandwidth
from sklearn.metrics import silhouette_score
from scipy.signal import savgol_filter, find_peaks
from umap import UMAP

# ============================================================
# MODEL
# ============================================================

class HierarchicalRAE(nn.Module):
    def __init__(self,
                 joint_dim,
                 joint_embed=32,
                 pose_embed=128,
                 hidden_dim=256,
                 latent_dim=30,
                 num_joints=7):
        super().__init__()
        self.num_joints = num_joints

        # --- Encoder ---
        self.joint_encoder = nn.Sequential(
            nn.Linear(joint_dim, joint_embed),
            nn.Tanh(),
            nn.Linear(joint_embed, joint_embed)
        )
        self.pose_encoder = nn.Sequential(
            nn.Linear(num_joints * joint_embed, pose_embed),
            nn.Tanh()
        )
        self.encoder_rnn = nn.LSTM(pose_embed, hidden_dim, batch_first=True)
        self.fc_latent    = nn.Linear(hidden_dim, latent_dim)

        # --- Decoder ---
        self.fc_decode_h  = nn.Linear(latent_dim, hidden_dim)
        self.fc_decode_c  = nn.Linear(latent_dim, hidden_dim)
        self.decoder_rnn  = nn.LSTM(latent_dim, hidden_dim, batch_first=True)
        self.decoder_proj = nn.Linear(hidden_dim, pose_embed)
        self.pose_decoder = nn.Sequential(
            nn.Linear(pose_embed, num_joints * joint_embed),
            nn.Tanh()
        )
        self.joint_decoder = nn.Linear(joint_embed, joint_dim)

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc_decode_h.weight, gain=2.0)
        nn.init.xavier_uniform_(self.fc_decode_c.weight, gain=2.0)
        nn.init.constant_(self.fc_decode_h.bias, 0.0)
        nn.init.constant_(self.fc_decode_c.bias, 0.0)
        nn.init.xavier_uniform_(self.fc_latent.weight, gain=2.0)
        nn.init.constant_(self.fc_latent.bias, 0.0)

        for name, param in self.decoder_rnn.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param, gain=2.0)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param, gain=2.0)
            elif 'bias' in name:
                nn.init.zeros_(param)

        for name, param in self.encoder_rnn.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)

        for module in [self.joint_encoder, self.pose_encoder,
                       self.pose_decoder, self.joint_decoder,
                       self.decoder_proj]:
            if isinstance(module, nn.Sequential):
                for layer in module:
                    if isinstance(layer, nn.Linear):
                        nn.init.xavier_uniform_(layer.weight)
                        nn.init.zeros_(layer.bias)
            elif isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def encode(self, x, lengths=None):
        from torch.nn.utils.rnn import pack_padded_sequence
        B, T, J, C = x.shape

        x_flat = x.view(B * T * J, C)
        x_flat = self.joint_encoder(x_flat)
        x_enc  = x_flat.view(B, T, J, -1)
        x_enc  = x_enc.view(B, T, -1)
        x_enc  = self.pose_encoder(x_enc)

        if lengths is not None:
            packed = pack_padded_sequence(
                x_enc, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            _, (h, _) = self.encoder_rnn(packed)
        else:
            _, (h, _) = self.encoder_rnn(x_enc)

        z = self.fc_latent(h[-1])
        return z

    def decode(self, z, T):
        B   = z.shape[0]
        h   = self.fc_decode_h(z).unsqueeze(0)
        c   = self.fc_decode_c(z).unsqueeze(0)
        inp = z.unsqueeze(1).repeat(1, T, 1)
        dec, _ = self.decoder_rnn(inp, (h, c))
        dec    = self.decoder_proj(dec)
        return dec

    def forward(self, x, lengths=None):
        B, T, J, C = x.shape
        z   = self.encode(x, lengths)
        dec = self.decode(z, T)
        dec = self.pose_decoder(dec)
        dec = dec.view(B, T, J, -1)
        dec = dec.view(B * T * J, -1)
        dec = self.joint_decoder(dec)
        dec = dec.view(B, T, J, C)
        return dec, z


# ============================================================
# COLLATE FUNCTION
# ============================================================

def collate_variable_length(batch):
    lengths = [x.shape[0] for x in batch]
    B       = len(batch)
    T_max   = max(lengths)
    J, C    = batch[0].shape[1], batch[0].shape[2]

    padded = torch.zeros(B, T_max, J, C)
    for i, x in enumerate(batch):
        padded[i, :lengths[i]] = x

    return padded, torch.tensor(lengths, dtype=torch.long)


# ============================================================
# EXTRACT WINDOWS (fixed-size, Stage 1 only)
# ============================================================

def extract_nonoverlapping_windows(raw_sequence, window_size):
    n_frames = len(raw_sequence)
    windows  = []
    for start in range(0, n_frames - window_size, window_size):
        windows.append(raw_sequence[start:start + window_size])
    windows = np.array(windows)
    print(f"Extracted {len(windows)} non-overlapping windows")
    print(f"Coverage: {len(windows) * window_size}/{n_frames} frames "
          f"({100 * len(windows) * window_size / n_frames:.1f}%)")
    return windows


# ============================================================
# STAGE 1: Train on fixed-size windows
# ============================================================

def train_on_fixed_windows(raw_sequence, window_size=60,
                            epochs=1000, batch_size=32,
                            lr=1e-4, device=device,
                            patience=30):
    windows  = extract_nonoverlapping_windows(raw_sequence, window_size)
    X_tensor = torch.tensor(windows, dtype=torch.float32)

    n_joints  = raw_sequence.shape[1]
    joint_dim = raw_sequence.shape[2]

    from torch.utils.data import TensorDataset
    dataset = TensorDataset(X_tensor)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model     = HierarchicalRAE(latent_dim=30, joint_dim=joint_dim,
                                num_joints=n_joints).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='min', factor=0.5, patience=10, verbose=False)
    loss_fn   = nn.MSELoss()

    best_loss    = float('inf')
    patience_ctr = 0
    best_weights = None
    lossgraph    = []

    print("Training on fixed windows...")
    for epoch in range(epochs):
        total_loss = 0
        for (batch,) in loader:
            batch    = batch.to(device)
            recon, _ = model(batch)
            loss     = loss_fn(recon, batch)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        avg = total_loss / len(loader)
        lossgraph.append(avg)
        scheduler.step(avg)

        if avg < best_loss - 1e-4:
            best_loss    = avg
            patience_ctr = 0
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1

        if patience_ctr >= patience:
            print(f"Early stopping at epoch {epoch+1} — best loss: {best_loss:.6f}")
            model.load_state_dict(best_weights)
            break

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} — Loss: {avg:.6f}  "
                  f"patience: {patience_ctr}/{patience}")

    return model, lossgraph


# ============================================================
# STAGE 4b: Retrain on variable-length windows
# ============================================================

def train_on_variable_windows(raw_sequence, windows,
                               epochs=1000, batch_size=32,
                               lr=1e-4, device=device,
                               patience=30):
    n_joints  = raw_sequence.shape[1]
    joint_dim = raw_sequence.shape[2]

    tensor_list = [torch.tensor(w, dtype=torch.float32) for w in windows]
    loader = DataLoader(tensor_list, batch_size=batch_size, shuffle=True,
                        collate_fn=collate_variable_length)

    model     = HierarchicalRAE(latent_dim=30, joint_dim=joint_dim,
                                num_joints=n_joints).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='min', factor=0.5, patience=10, verbose=False)
    loss_fn   = nn.MSELoss()

    best_loss    = float('inf')
    patience_ctr = 0
    best_weights = None
    lossgraph    = []

    print("Retraining on variable-length windows...")
    for epoch in range(epochs):
        total_loss = 0
        for padded, lengths in loader:
            padded   = padded.to(device)
            recon, _ = model(padded, lengths=lengths)

            loss = torch.tensor(0.0, device=device)
            for i, l in enumerate(lengths):
                loss = loss + loss_fn(recon[i, :l], padded[i, :l])
            loss = loss / len(lengths)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        avg = total_loss / len(loader)
        lossgraph.append(avg)
        scheduler.step(avg)

        if avg < best_loss - 1e-4:
            best_loss    = avg
            patience_ctr = 0
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1

        if patience_ctr >= patience:
            print(f"Early stopping at epoch {epoch+1} — best loss: {best_loss:.6f}")
            model.load_state_dict(best_weights)
            break

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} — Loss: {avg:.6f}  "
                  f"patience: {patience_ctr}/{patience}")

    return model, lossgraph

# ============================================================
# STAGE 2: Compute reconstruction loss signal
# ============================================================

def compute_frame_loss(model, raw_sequence, window_size,
                        stride=5, device=device):
    model.eval()
    losses    = []
    positions = []

    raw_tensor = torch.tensor(raw_sequence, dtype=torch.float32, device=device)
    loss_fn    = nn.MSELoss()

    with torch.no_grad():
        for start in range(0, len(raw_sequence) - window_size, stride):
            window   = raw_tensor[start:start + window_size].unsqueeze(0)
            recon, _ = model(window)
            loss     = loss_fn(recon, window).item()
            losses.append(loss)
            positions.append(start + window_size // 2)

    positions = np.array(positions)
    losses    = np.array(losses)
    print(f"Computed loss at {len(losses)} positions")
    print(f"Loss stats — mean: {losses.mean():.4f}, "
          f"std: {losses.std():.4f}, max: {losses.max():.4f}")
    return positions, losses


# ============================================================
# STAGE 3: Detect transitions
# ============================================================

def find_transitions(positions, losses,
                      percentile=70, smoothing=5,
                      min_distance=3, fps=30):
    if len(losses) < smoothing:
        smoothing = max(3, len(losses) // 2)
        if smoothing % 2 == 0:
            smoothing += 1

    smoothed  = savgol_filter(losses, window_length=smoothing, polyorder=2)
    threshold = np.percentile(smoothed, percentile)
    peaks, _  = find_peaks(smoothed, height=threshold, distance=min_distance)
    transition_frames = positions[peaks]

    if len(transition_frames) > 1:
        intervals = np.diff(transition_frames)
        mean_bout = np.mean(intervals) / fps
        print(f"Found {len(transition_frames)} transitions")
        print(f"Mean bout duration: {mean_bout:.2f}s")
        if mean_bout < 1:
            print("WARNING: bouts too short — raise percentile or min_distance")
        elif mean_bout > 15:
            print("WARNING: bouts too long — lower percentile")
        else:
            print("Bout duration looks plausible")

    return transition_frames, smoothed


# ============================================================
# STAGE 4: Variable-length windows from segments
# ============================================================

def create_windows_from_transitions(raw_sequence, transition_frames,
                                     min_segment_frames=15,
                                     max_segment_frames=300):
    n_frames   = len(raw_sequence)
    boundaries = np.unique(
        np.concatenate([[0], transition_frames, [n_frames]])
    ).astype(int)

    all_windows   = []
    window_labels = []

    for seg_idx in range(len(boundaries) - 1):
        seg_start = boundaries[seg_idx]
        seg_end   = boundaries[seg_idx + 1]
        seg_len   = seg_end - seg_start

        if seg_len < min_segment_frames:
            continue

        segment = raw_sequence[seg_start:seg_end]

        for start in range(0, seg_len, max_segment_frames):
            chunk = segment[start:start + max_segment_frames]
            if len(chunk) < min_segment_frames:
                continue
            all_windows.append(chunk)
            window_labels.append(seg_idx)

    lengths = [len(w) for w in all_windows]
    print(f"Created {len(all_windows)} variable-length windows from "
          f"{len(boundaries) - 1} segments")
    print(f"Window lengths — min: {min(lengths)}, "
          f"max: {max(lengths)}, mean: {np.mean(lengths):.1f}")

    return all_windows, np.array(window_labels)


# ============================================================
# STAGE 5: Encode, UMAP, cluster
# ============================================================

def encode_and_cluster(model, windows, batch_size=32,
                        device='cpu', quantile=0.1,
                        umap_neighbors=30, umap_min_dist=0.1):
    model.eval()
    all_latents = []

    tensor_list = [torch.tensor(w, dtype=torch.float32) for w in windows]
    loader = DataLoader(tensor_list, batch_size=batch_size, shuffle=False,
                        collate_fn=collate_variable_length)

    with torch.no_grad():
        for padded, lengths in loader:
            padded = padded.to(device)
            _, z   = model(padded, lengths=lengths)
            all_latents.append(z.cpu().numpy())

    all_latents = np.concatenate(all_latents, axis=0)  # (N, 16)
    print(f"Latents shape: {all_latents.shape}")

    # UMAP: 16D → 2D
    print("Running UMAP...")
    reducer    = UMAP(n_components=2, n_neighbors=umap_neighbors,
                      min_dist=umap_min_dist, metric='cosine',
                      random_state=42)
    latents_2d = reducer.fit_transform(all_latents)     # (N, 2)
    print(f"UMAP done. Shape: {latents_2d.shape}")

    # MeanShift on 2D UMAP embedding
    normed    = normalize(latents_2d, norm='l2')
    bandwidth = estimate_bandwidth(normed, quantile=quantile)
    print(f"Estimated bandwidth: {bandwidth:.4f}")

    ms     = MeanShift(bandwidth=bandwidth, bin_seeding=True)
    ms.fit(normed)
    labels = ms.labels_

    n_clusters = len(np.unique(labels))
    print(f"Found {n_clusters} clusters")
    print(f"Cluster sizes: {np.bincount(labels)}")

    if n_clusters > 1:
        dist_matrix = np.clip(1 - np.dot(normed, normed.T), 0, 2)
        sil = silhouette_score(dist_matrix, labels, metric='precomputed')
        print(f"Silhouette score: {sil:.4f}")

    return all_latents, latents_2d, labels, ms


# ============================================================
# FULL TWO-PASS PIPELINE
# ============================================================

def run_pipeline_test(raw_sequence, window_size, fps,
                      epochs, percentile, quantile,
                      min_segment_frames, max_segment_frames,
                      stride=5, umap_neighbors=30,
                      umap_min_dist=0.1, device=device):

    # ── PASS 1: fixed windows → transition detection ───────────────────
    print("\n" + "="*50)
    print("STAGE 1: Training RAE on fixed windows")
    print("="*50)
    model_stage1, lossgraph_stage1 = train_on_fixed_windows(
        raw_sequence, window_size=window_size,
        epochs=epochs, device=device
    )

    print("\n" + "="*50)
    print("STAGE 2: Computing reconstruction loss signal")
    print("="*50)
    positions, losses = compute_frame_loss(
        model_stage1, raw_sequence, window_size,
        stride=stride, device=device
    )

    print("\n" + "="*50)
    print("STAGE 3: Finding behavioral transitions")
    print("="*50)
    transition_frames, smoothed = find_transitions(
        positions, losses,
        percentile=percentile, fps=fps
    )

    print("\n" + "="*50)
    print("STAGE 4: Creating variable-length behavioral windows")
    print("="*50)
    windows, window_segment_labels = create_windows_from_transitions(
        raw_sequence, transition_frames,
        min_segment_frames=min_segment_frames,
        max_segment_frames=max_segment_frames
    )

    # ── PASS 2: retrain on variable-length windows ─────────────────────
    print("\n" + "="*50)
    print("STAGE 4b: Retraining RAE on variable-length windows")
    print("="*50)
    model_stage2, lossgraph_stage2 = train_on_variable_windows(
        raw_sequence, windows,
        epochs=epochs, device=device
    )

    print("\n" + "="*50)
    print("STAGE 5: Encoding + UMAP + clustering")
    print("="*50)
    latents, latents_2d, cluster_labels, ms_model = encode_and_cluster(
        model_stage2, windows,
        quantile=quantile,
        umap_neighbors=umap_neighbors,
        umap_min_dist=umap_min_dist,
        device=device
    )

    return {
        'model':                 model_stage2,
        'model_stage1':          model_stage1,
        'latents':               latents,        # (N, 16) — raw high-dim latents
        'latents_2d':            latents_2d,     # (N, 2)  — UMAP projection
        'cluster_labels':        cluster_labels,
        'transition_frames':     transition_frames,
        'windows':               windows,
        'window_segment_labels': window_segment_labels,
        'losses':                losses,
        'positions':             positions,
        'lossgraph_stage1':      lossgraph_stage1,
        'lossgraph_stage2':      lossgraph_stage2,
        'smoothed_losses':       smoothed,
    }

In [11]:
# ============================================================
# HYPERPARAMETER TUNING: Bayesian optimization over percentile, quantile, & lr
# ============================================================
import gc
import numpy as np
from sklearn.metrics import silhouette_score
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args

MAX_ITER = 15
PERCENTILE_RANGE = (45.0, 70.0)
QUANTILE_RANGE = (0.05, 0.2)
LR_RANGE = (1e-4, 1e-2)

# ── Train Stage 1 ONCE (shared across all iterations) ────────
print("="*60)
print("PRE-STEP: Training Stage 1 model (shared across iterations)")
print("="*60)
model_stage1, lossgraph_stage1 = train_on_fixed_windows(
    raw_processed, window_size=60, epochs=1000
)

print("\n" + "="*60)
print("PRE-STEP: Computing reconstruction loss signal (shared)")
print("="*60)
positions, losses = compute_frame_loss(
    model_stage1, raw_processed, window_size=60, stride=5, device=device
)

# ── Define search space ──────────────────────────────────────
search_space = [
    Real(*PERCENTILE_RANGE, name='percentile'),
    Real(*QUANTILE_RANGE,   name='quantile'),
    Real(*LR_RANGE,         name='lr', prior='log-uniform'),
]

search_log = []
best_results = None
best_score = -1
iteration = 0

# ── Objective function ───────────────────────────────────────
@use_named_args(search_space)
def objective(percentile, quantile, lr):
    global iteration, best_score, best_results

    iteration += 1
    print(f"\n{'='*60}")
    print(f"ITERATION {iteration}/{MAX_ITER}  |  percentile={percentile:.2f}, quantile={quantile:.4f}, lr={lr:.6f}")
    print(f"{'='*60}")

    # Stage 3: transitions
    transition_frames, smoothed = find_transitions(
        positions, losses, percentile=percentile, fps=30
    )

    # Stage 4: variable windows
    windows, window_segment_labels = create_windows_from_transitions(
        raw_processed, transition_frames,
        min_segment_frames=60, max_segment_frames=600
    )

    # Stage 4b: retrain on variable windows (with tuned lr)
    model_stage2, lossgraph_stage2 = train_on_variable_windows(
        raw_processed, windows, epochs=2000, lr=lr, device=device
    )

    # Stage 5: encode + cluster
    latents, latents_2d, cluster_labels, ms_model = encode_and_cluster(
        model_stage2, windows, quantile=quantile,
        umap_neighbors=30, umap_min_dist=0.1, device=device
    )

    n_clusters = len(np.unique(cluster_labels))

    # ── Compute silhouette score ──────────────────────────────
    # silhouette needs at least 2 clusters and more samples than clusters
    if n_clusters < 2 or n_clusters >= len(latents_2d):
        print(f"  → {n_clusters} clusters — skipping silhouette (invalid cluster count)")
        score = -1.0
    else:
        score = silhouette_score(latents_2d, cluster_labels)

    log_entry = {
        "iteration": iteration,
        "percentile": round(percentile, 2),
        "quantile": round(quantile, 4),
        "lr": round(float(lr), 6),
        "n_clusters": n_clusters,
        "silhouette": round(float(score), 4),
    }
    search_log.append(log_entry)
    print(f"  → {n_clusters} clusters, silhouette={score:.4f}")

    # Track best
    if score > best_score:
        best_score = score
        best_results = {
            'model': model_stage2,
            'model_stage1': model_stage1,
            'latents': latents,
            'latents_2d': latents_2d,
            'cluster_labels': cluster_labels,
            'transition_frames': transition_frames,
            'windows': windows,
            'window_segment_labels': window_segment_labels,
            'losses': losses,
            'positions': positions,
            'lossgraph_stage1': lossgraph_stage1,
            'lossgraph_stage2': lossgraph_stage2,
            'smoothed_losses': smoothed,
            '_percentile': percentile,
            '_quantile': quantile,
            '_lr': lr,
        }
        print(f"  ** NEW BEST (silhouette={score:.4f}) **")

    # Cleanup GPU memory
    del model_stage2, latents, latents_2d, cluster_labels, ms_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # gp_minimize minimizes, so return negative score
    return -score

# ── Run Bayesian optimization ────────────────────────────────
bayes_result = gp_minimize(
    func=objective,
    dimensions=search_space,
    n_calls=MAX_ITER,
    n_initial_points=5,
    random_state=42,
    verbose=False,
)

# ── Summary ──────────────────────────────────────────────────
print(f"\n{'='*60}")
print("HYPERPARAMETER SEARCH COMPLETE")
print(f"{'='*60}")
print(f"\nSearch log:")
for entry in search_log:
    marker = " <-- BEST" if entry['silhouette'] == best_score else ""
    print(f"  Iter {entry['iteration']:2d}: percentile={entry['percentile']:5.2f}, "
          f"quantile={entry['quantile']:.4f}, lr={entry['lr']:.6f}  "
          f"→  {entry['n_clusters']} clusters, silhouette={entry['silhouette']:.4f}{marker}")

print(f"\nBest result:")
print(f"  percentile : {best_results['_percentile']:.2f}")
print(f"  quantile   : {best_results['_quantile']:.4f}")
print(f"  lr         : {best_results['_lr']:.6f}")
print(f"  clusters   : {len(np.unique(best_results['cluster_labels']))}")
print(f"  silhouette : {best_score:.4f}")

# Set results to best for downstream cells
results = best_results

PRE-STEP: Training Stage 1 model (shared across iterations)
Extracted 62621 non-overlapping windows
Coverage: 3757260/3757302 frames (100.0%)
Training on fixed windows...


c:\Users\ariAccount\anaconda3\envs\pipeline_env\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 10/1000 — Loss: 0.598149  patience: 0/30
Epoch 20/1000 — Loss: 0.579368  patience: 0/30
Epoch 30/1000 — Loss: 0.563062  patience: 1/30
Epoch 40/1000 — Loss: 0.549756  patience: 1/30
Epoch 50/1000 — Loss: 0.535771  patience: 0/30
Epoch 60/1000 — Loss: 0.523634  patience: 0/30
Epoch 70/1000 — Loss: 0.513415  patience: 0/30
Epoch 80/1000 — Loss: 0.500890  patience: 0/30
Epoch 90/1000 — Loss: 0.489322  patience: 0/30
Epoch 100/1000 — Loss: 0.488199  patience: 3/30
Epoch 110/1000 — Loss: 0.479790  patience: 2/30
Epoch 120/1000 — Loss: 0.477241  patience: 1/30
Epoch 130/1000 — Loss: 0.467400  patience: 1/30
Epoch 140/1000 — Loss: 0.467482  patience: 1/30
Epoch 150/1000 — Loss: 0.456803  patience: 0/30
Epoch 160/1000 — Loss: 0.452034  patience: 1/30
Epoch 170/1000 — Loss: 0.446143  patience: 0/30
Epoch 180/1000 — Loss: 0.439286  patience: 0/30
Epoch 190/1000 — Loss: 0.437743  patience: 4/30
Epoch 200/1000 — Loss: 0.431889  patience: 0/30
Epoch 210/1000 — Loss: 0.431744  patience: 3/30
E

c:\Users\ariAccount\anaconda3\envs\pipeline_env\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Retraining on variable-length windows...
Epoch 10/2000 — Loss: 0.300303  patience: 1/30
Epoch 20/2000 — Loss: 0.291547  patience: 0/30
Epoch 30/2000 — Loss: 0.288308  patience: 2/30
Epoch 40/2000 — Loss: 0.285856  patience: 2/30
Epoch 50/2000 — Loss: 0.283547  patience: 1/30
Epoch 60/2000 — Loss: 0.283915  patience: 1/30
Epoch 70/2000 — Loss: 0.282991  patience: 1/30
Epoch 80/2000 — Loss: 0.281793  patience: 11/30
Epoch 90/2000 — Loss: 0.274448  patience: 1/30
Epoch 100/2000 — Loss: 0.273178  patience: 2/30
Epoch 110/2000 — Loss: 0.272643  patience: 6/30
Epoch 120/2000 — Loss: 0.272068  patience: 0/30
Epoch 130/2000 — Loss: 0.272467  patience: 4/30
Epoch 140/2000 — Loss: 0.271820  patience: 8/30
Epoch 150/2000 — Loss: 0.266115  patience: 1/30
Epoch 160/2000 — Loss: 0.265038  patience: 4/30
Epoch 170/2000 — Loss: 0.264763  patience: 5/30
Epoch 180/2000 — Loss: 0.259741  patience: 2/30
Epoch 190/2000 — Loss: 0.259040  patience: 2/30
Epoch 200/2000 — Loss: 0.257825  patience: 0/30
Epoch 2

...

c:\Users\ariAccount\anaconda3\envs\pipeline_env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP done. Shape: (11908, 2)
Estimated bandwidth: 0.0753
Found 6 clusters
Cluster sizes: [3480 2790 2281 1642  881  834]
Silhouette score: 0.6939
  → 6 clusters, silhouette=0.2191
  ** NEW BEST (silhouette=0.2191) **

ITERATION 3/15  |  percentile=56.48, quantile=0.1001, lr=0.000193
Found 47283 transitions
Mean bout duration: 2.65s
Bout duration looks plausible
Created 12184 variable-length windows from 47284 segments
Window lengths — min: 60, max: 600, mean: 218.2


c:\Users\ariAccount\anaconda3\envs\pipeline_env\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Retraining on variable-length windows...
Epoch 10/2000 — Loss: 0.293517  patience: 0/30
Epoch 20/2000 — Loss: 0.288529  patience: 0/30
Epoch 30/2000 — Loss: 0.281602  patience: 0/30
Epoch 40/2000 — Loss: 0.272169  patience: 0/30
Epoch 50/2000 — Loss: 0.267397  patience: 0/30
Epoch 60/2000 — Loss: 0.264279  patience: 1/30
Epoch 70/2000 — Loss: 0.262338  patience: 2/30
Epoch 80/2000 — Loss: 0.259185  patience: 0/30
Epoch 90/2000 — Loss: 0.258120  patience: 0/30
Epoch 100/2000 — Loss: 0.256697  patience: 2/30
Epoch 110/2000 — Loss: 0.255242  patience: 3/30
Epoch 120/2000 — Loss: 0.253606  patience: 3/30
Epoch 130/2000 — Loss: 0.252769  patience: 1/30
Epoch 140/2000 — Loss: 0.251484  patience: 6/30
Epoch 150/2000 — Loss: 0.250309  patience: 0/30
Epoch 160/2000 — Loss: 0.249970  patience: 1/30
Epoch 170/2000 — Loss: 0.248388  patience: 2/30
Epoch 180/2000 — Loss: 0.247528  patience: 1/30
Epoch 190/2000 — Loss: 0.246583  patience: 6/30
Epoch 200/2000 — Loss: 0.244874  patience: 6/30
Epoch 21

KeyboardInterrupt: 

results = run_pipeline_test(
    raw_processed, 
    window_size=30,
    fps=30,
    epochs=300,
    percentile=51.5,
    quantile=0.154,
    min_segment_frames=30, 
    max_segment_frames=600,
    stride = 5,
    device='cpu'
)

In [ ]:
embedded = results['latents']
labels = results['cluster_labels']
plt.figure(figsize=(8, 6))
plt.scatter(embedded[:, 0], embedded[:, 1])
plt.title("Latent Space")
plt.xlabel("1")
plt.ylabel("2")
plt.show()


"""latents = results['latents']
normed  = normalize(latents, norm='l2')
labels = results['cluster_labels']
# UMAP to 2D
reducer   = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.5, random_state=42)
embedded  = reducer.fit_transform(normed)
"""



embedded = results['latents']
labels = results['cluster_labels']
plt.figure(figsize=(8, 6))
plt.scatter(embedded[:, 0], embedded[:, 1],
            c=labels, cmap='tab10', s=15, alpha=0.8)
plt.title("Clusters: quantile=0.17, n_clusters={5}")
plt.colorbar(label='Cluster')
plt.xlabel("1")
plt.ylabel("2")
plt.show()

In [ ]:
plt.plot(best_results['lossgraph_stage1'], label='stage 1')
plt.plot(best_results['lossgraph_stage2'], label='stage 2')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

In [ ]:
ground_truth_labels = np.load("rat_movement_large_labels.npy", allow_pickle = True)
raw_sequence = np.load("rat_movement_large.npy")
from scipy.stats import mode

In [ ]:
gt_window_labels = []
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy.optimize import linear_sum_assignment

# ── Parameters (must match pipeline) ──────────────────────────────────────
min_segment_frames = 30
max_segment_frames = 600

# ── Reconstruct window frame ranges ───────────────────────────────────────
n_frames   = len(raw_sequence)
boundaries = np.unique(
    np.concatenate([[0], results['transition_frames'], [n_frames]])
).astype(int)

gt_window_labels = []
for seg_idx in range(len(boundaries) - 1):
    seg_start = boundaries[seg_idx]
    seg_end   = boundaries[seg_idx + 1]
    seg_len   = seg_end - seg_start

    if seg_len < min_segment_frames:
        continue

    for chunk_start in range(0, seg_len, max_segment_frames):
        abs_start = seg_start + chunk_start
        abs_end   = min(abs_start + max_segment_frames, seg_end)
        chunk_len = abs_end - abs_start

        if chunk_len < min_segment_frames:
            continue

        window_frames = ground_truth_labels[abs_start:abs_end]
        values, counts = np.unique(window_frames, return_counts=True)
        majority = values[np.argmax(counts)]
        gt_window_labels.append(majority)

gt_window_labels = np.array(gt_window_labels)
cluster_labels   = results['cluster_labels']

print(f"GT labels:      {len(gt_window_labels)}")
print(f"Cluster labels: {len(cluster_labels)}")
assert len(gt_window_labels) == len(cluster_labels), \
    f"Mismatched: {len(gt_window_labels)} vs {len(cluster_labels)}"

# ── Convert GT strings to numeric for sklearn metrics ─────────────────────
behaviour_names = np.unique(gt_window_labels)   # e.g. ['explore' 'groom' ...]
beh_to_idx      = {b: i for i, b in enumerate(behaviour_names)}
gt_numeric      = np.array([beh_to_idx[b] for b in gt_window_labels])

# ── Metrics ────────────────────────────────────────────────────────────────
ari = adjusted_rand_score(gt_numeric, cluster_labels)
nmi = normalized_mutual_info_score(gt_numeric, cluster_labels)
print(f"\nARI:    {ari:.3f}  (1.0 = perfect, 0 = random)")
print(f"NMI:    {nmi:.3f}  (1.0 = perfect, 0 = random)")

# ── Build confusion matrix manually (GT rows, cluster columns) ────────────
cluster_ids = np.unique(cluster_labels)
cm = np.zeros((len(behaviour_names), len(cluster_ids)), dtype=int)

for gt, pred in zip(gt_window_labels, cluster_labels):
    cm[beh_to_idx[gt], pred] += 1

# ── Hungarian matching: optimal cluster → behaviour assignment ─────────────
row_ind, col_ind = linear_sum_assignment(-cm)
print("\nOptimal cluster → behaviour mapping:")
for r, c in zip(row_ind, col_ind):
    total    = cm[:, c].sum()
    correct  = cm[r, c]
    print(f"  Cluster {c:2d}  →  {behaviour_names[r]:14s}  "
          f"({correct}/{total} = {correct/total:.1%})")

# ── Purity ─────────────────────────────────────────────────────────────────
purity = np.sum(np.max(cm, axis=0)) / np.sum(cm)
print(f"\nPurity: {purity:.3f}")

# ── Pure windows ───────────────────────────────────────────────────────────
pure_windows = []
for seg_idx in range(len(boundaries) - 1):
    seg_start = boundaries[seg_idx]
    seg_end   = boundaries[seg_idx + 1]
    seg_len   = seg_end - seg_start

    if seg_len < min_segment_frames:
        continue

    for chunk_start in range(0, seg_len, max_segment_frames):
        abs_start = seg_start + chunk_start
        abs_end   = min(abs_start + max_segment_frames, seg_end)
        if (abs_end - abs_start) < min_segment_frames:
            continue
        n_unique = len(np.unique(ground_truth_labels[abs_start:abs_end]))
        pure_windows.append(n_unique == 1)

print(f"Pure windows:   {np.mean(pure_windows):.1%}")

# ── Plot confusion matrix ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
            xticklabels=[f'Cluster {i}' for i in cluster_ids],
            yticklabels=behaviour_names)
ax.set_xlabel("Predicted Cluster")
ax.set_ylabel("Ground Truth Behaviour")
ax.set_title(f"Confusion Matrix  (ARI={ari:.3f},  NMI={nmi:.3f},  Purity={purity:.3f})")
plt.tight_layout()
plt.show()

In [ ]:
# ── Check 1: how pure are your windows? ───────────────────
# if this is low, transition detection is the bottleneck
print(f"Pure windows: {np.mean(pure_windows):.1%}")
# if < 70%, fix transition detection before anything else

# ── Check 2: does the latent space have structure? ─────────
# plot GT labels on the latent space
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
scatter = axes[0].scatter(results['latents'][:, 0], results['latents'][:, 1],
                           c=gt_numeric, cmap='tab10', alpha=0.7, s=20)
axes[0].set_title("Ground Truth Behaviours")
plt.colorbar(scatter, ax=axes[0],
             ticks=range(len(behaviour_names))).set_ticklabels(behaviour_names)

scatter2 = axes[1].scatter(results['latents'][:, 0], results['latents'][:, 1],
                            c=results['cluster_labels'], cmap='tab10', alpha=0.7, s=20)
axes[1].set_title("Predicted Clusters")
plt.colorbar(scatter2, ax=axes[1])
plt.tight_layout()
plt.show()
# if GT colours are jumbled → model isn't learning, fix preprocessing/training
# if GT colours show structure but clusters don't align → fix clustering

# ── Check 3: upper bound ARI if windows were perfect ───────
# assign each window its majority GT label as the prediction
# this tells you the maximum ARI your transition detector allows
from sklearn.metrics import adjusted_rand_score
upper_bound_ari = adjusted_rand_score(gt_numeric, gt_numeric)
print(f"Upper bound ARI (perfect clustering): {upper_bound_ari:.3f}")  # should be 1.0

# more useful — what ARI would you get if you clustered perfectly
# on only the pure windows?
pure_mask = np.array(pure_windows)
if pure_mask.sum() > 0:
    ari_pure = adjusted_rand_score(gt_numeric[pure_mask],
                                   results['cluster_labels'][pure_mask])
    print(f"ARI on pure windows only: {ari_pure:.3f}")